# LeetCode 3145 — Find Products of Elements of Big Array bằng Python

- Link: https://leetcode.com/problems/find-products-of-elements-of-big-array/
- Độ khó: **Hard**
- Chủ đề: Mọi phần tử của `big_nums` là lũy thừa của 2 ⇒ tích = `2^(tổng số mũ)`; đếm **số bit 1** và **tổng vị trí bit 1** của các số `1..x` theo từng bit (công thức chu kỳ), **tìm nhị phân** số `x` chứa chỉ số cần tìm, rồi `pow(2, E, mod)`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findProductsOfElements(self, queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. "Mảng mạnh" của `x` = các lũy thừa 2 ứng với bit 1 của `x`, sắp tăng dần ⇒ độ dài = `popcount(x)`.
2. Tích các phần tử `= 2^E` với `E` = tổng **số mũ** (vị trí bit) ⇒ chỉ cần hàm `E(N)` = tổng số mũ của `N` phần tử đầu, rồi đáp án `pow(2, E(to+1) − E(from), mod)`.
3. Đếm số lần bit `b` bật trong `0..x` bằng chu kỳ `2^{b+1}`: `((x+1) >> (b+1)) · 2^b + max(0, (x+1) mod 2^{b+1} − 2^b)`.
4. Tìm `x` lớn nhất có `Σ_{i ≤ x} popcount(i) ≤ N` bằng tìm nhị phân; phần dư lấy các bit **thấp nhất** của `x + 1`.
5. Không dùng định lý Fermat để giảm số mũ (`mod` không nhất thiết nguyên tố) — `pow` ba đối số xử lý số mũ lớn trực tiếp.

## 2. Đề bài & ví dụ

**Mảng mạnh** của số nguyên dương `x` là mảng sắp tăng ngắn nhất gồm các lũy thừa của 2 có tổng `x` (ví dụ `10 → [2, 8]`, `13 → [1, 4, 8]`). `big_nums` là nối các mảng mạnh của `1, 2, 3, …`: `[1, 2, 1, 2, 4, 1, 4, 2, 4, 1, 2, 4, 8, …]` (chỉ số từ 0).

Với truy vấn `[from, to, mod]`, tính `(big_nums[from] · … · big_nums[to]) mod mod`.

**Ví dụ 1:** `[[1,3,7]]` → `[4]`. **Ví dụ 2:** `[[2,5,3],[7,7,4]]` → `[2,2]`.

**Ràng buộc:** `1 ≤ queries.length ≤ 500`; `0 ≤ from ≤ to ≤ 10^15`; `1 ≤ mod ≤ 10^5`.

In [1]:
from typing import List
import random, time

examples = [([[1, 3, 7]], [4]), ([[2, 5, 3], [7, 7, 4]], [2, 2])]
def powerful(x):
    return [1 << b for b in range(x.bit_length()) if x >> b & 1]

for x in (1, 8, 10, 13, 23):
    print(x, bin(x), '->', powerful(x))
big = [v for x in range(1, 9) for v in powerful(x)]
print('big_nums đầu tiên:', big)

1 0b1 -> [1]
8 0b1000 -> [8]
10 0b1010 -> [2, 8]
13 0b1101 -> [1, 4, 8]
23 0b10111 -> [1, 2, 4, 16]
big_nums đầu tiên: [1, 2, 1, 2, 4, 1, 4, 2, 4, 1, 2, 4, 8]


## 3. Kiến thức nền

### 3.1. Đếm bit theo chu kỳ

Trong dãy `0, 1, 2, …, x`, bit `b` có chu kỳ `2^{b+1}`: `2^b` số 0 rồi `2^b` số 1. Với `L = x + 1` số:

`ones_b(x) = (L >> (b+1)) · 2^b + max(0, (L mod 2^{b+1}) − 2^b)`.

Từ đó `C(x) = Σ_b ones_b(x)` = **số phần tử** do `1..x` góp vào `big_nums`, và `S(x) = Σ_b b · ones_b(x)` = **tổng số mũ** của các phần tử đó.

In [2]:
def ones_bit(x, b):
    L = x + 1
    return (L >> (b + 1) << b) + max(0, (L & ((1 << (b + 1)) - 1)) - (1 << b))

def C(x):
    return sum(ones_bit(x, b) for b in range(x.bit_length() + 1)) if x > 0 else 0

def S(x):
    return sum(b * ones_bit(x, b) for b in range(x.bit_length() + 1)) if x > 0 else 0

for x in (1, 5, 13, 100):
    print(x, '| C =', C(x), sum(bin(i).count('1') for i in range(x + 1)),
          '| S =', S(x), sum(b for i in range(x + 1) for b in range(i.bit_length()) if i >> b & 1))

1 | C = 1 1 | S = 0 0
5 | C = 7 7 | S = 6 6
13 | C = 25 25 | S = 36 36
100 | C = 319 319 | S = 891 891


### 3.2. Tổng số mũ của `N` phần tử đầu

Gọi `E(N)` = tổng số mũ của `big_nums[0..N−1]`. Tìm `x` lớn nhất với `C(x) ≤ N` (các số `1..x` góp trọn). Còn `r = N − C(x) < popcount(x+1)` phần tử lấy từ mảng mạnh của `x + 1` — là `r` bit **thấp nhất** (mảng sắp tăng). Vậy `E(N) = S(x) + (tổng vị trí của r bit thấp nhất của x + 1)`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đừng nhân — **cộng số mũ**. Tích một đoạn `= 2^{E(to+1) − E(from)}`. Mọi khó khăn dồn vào việc "đếm bit 1 của mọi số tới `x`" khi `x` cỡ `10^{13}`, mà việc này làm theo từng bit trong `O(log x)`.

Ví dụ 1: `big_nums[1..3] = [2, 1, 2]` ⇒ số mũ `1 + 0 + 1 = 2` ⇒ `2² mod 7 = 4`.

## 5. Cách 1 — Brute Force: dựng `big_nums` thật

In [3]:
def brute(queries):
    need = max(q[1] for q in queries) + 1
    arr = []; x = 1
    while len(arr) < need:
        arr.extend(powerful(x)); x += 1
    res = []
    for fr, to, mod in queries:
        p = 1 % mod
        for i in range(fr, to + 1):
            p = p * arr[i] % mod
        res.append(p)
    return res

for q, exp in examples:
    print('brute:', brute(q), '| kỳ vọng', exp)

brute: [4] | kỳ vọng [4]
brute: [2, 2] | kỳ vọng [2, 2]


Chỉ dùng được khi `to` nhỏ (vài triệu). Lưu ý `1 % mod` để đúng khi `mod = 1`.

## 6. Ý tưởng thuật toán / bước đột phá

```
E(N):  x = max{ x : C(x) <= N } (tìm nhị phân)
       r = N − C(x); e = S(x) + tổng r vị trí bit thấp nhất của (x+1)
ans  = pow(2, E(to + 1) − E(from), mod)
```

Mỗi `E` tốn `O(log² X)` (tìm nhị phân × đếm bit), `X ≈ 10^{15} / 20`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tìm nhị phân + đếm bit theo chu kỳ (lời giải chính)

In [4]:
def E_binary(N):
    lo, hi = 0, 1
    while C(hi) <= N:
        hi *= 2
    while lo < hi:                       # x lớn nhất có C(x) <= N
        mid = (lo + hi + 1) // 2
        if C(mid) <= N:
            lo = mid
        else:
            hi = mid - 1
    x = lo; r = N - C(x); e = S(x); y = x + 1; b = 0
    while r:
        if y >> b & 1:
            e += b; r -= 1
        b += 1
    return e

def solve_binary(queries):
    return [pow(2, E_binary(to + 1) - E_binary(fr), mod) for fr, to, mod in queries]

for q, exp in examples:
    print('tìm nhị phân:', solve_binary(q), '| kỳ vọng', exp)

tìm nhị phân: [4] | kỳ vọng [4]
tìm nhị phân: [2, 2] | kỳ vọng [2, 2]


### 7.2. Cách 3 — Dựng `x` từng bit từ cao xuống (không tìm nhị phân)

Ý tưởng "digit DP": duyệt bit `b` từ cao xuống, giữ `cnt` = số bit 1 đã cố định ở phần đầu. Nếu đặt bit `b` của `x` bằng 1, ta "bỏ qua" khối `2^b` số có tiền tố hiện tại và bit `b = 0`: khối đó góp `cnt · 2^b + b · 2^{b−1}` phần tử (mỗi số có `cnt` bit cố định cộng trung bình `b/2` bit thấp) và `S` tương ứng. Nếu còn đủ "ngân sách" `N` thì nhận khối. Kết thúc, `x` là số đầu tiên chưa trọn, cộng thêm các bit thấp nhất của nó.

In [5]:
def E_bitwise(N):
    # đếm trên các số 0..(x-1) với x dựng dần; số 0 không góp gì
    x = 0; cnt = 0; possum = 0; used = 0; e = 0
    for b in range(55, -1, -1):
        # khối: các số có tiền tố (bit > b) như x, bit b = 0, bit thấp tuỳ ý: 2^b số
        blk = cnt * (1 << b) + (b * (1 << b) >> 1)                   # số phần tử của khối
        blk_e = possum * (1 << b) + ((b * (b - 1) // 2) * (1 << b) >> 1 if b >= 1 else 0)
        if used + blk <= N:
            used += blk; e += blk_e
            x |= 1 << b; cnt += 1; possum += b
    # x: số đầu tiên chưa góp trọn; phần còn lại là các bit thấp nhất của x
    r = N - used; b = 0
    while r:
        if x >> b & 1:
            e += b; r -= 1
        b += 1
    return e

def solve_bitwise(queries):
    return [pow(2, E_bitwise(to + 1) - E_bitwise(fr), mod) for fr, to, mod in queries]

for q, exp in examples:
    print('dựng từng bit:', solve_bitwise(q), '| kỳ vọng', exp)

dựng từng bit: [4] | kỳ vọng [4]
dựng từng bit: [2, 2] | kỳ vọng [2, 2]


Giải thích công thức khối: `2^b` số với `b` bit thấp tuỳ ý có tổng `b · 2^{b−1}` bit 1 ở phần thấp và tổng vị trí `(0 + 1 + … + (b−1)) · 2^{b−1} = b(b−1)/2 · 2^{b−1}`; cộng phần cố định `cnt` bit (tổng vị trí `possum`) cho mỗi số.

### 7.3. Cách 4 — Tiền xử lý tiền tố số mũ cho chỉ số nhỏ

Khi mọi `to` nhỏ (ví dụ `≤ 10^6`), dựng mảng tiền tố số mũ một lần rồi trả lời `O(1)` + `pow`. Dùng để đối chiếu nhanh trên nhiều truy vấn ngẫu nhiên.

In [6]:
def solve_prefix(queries):
    need = max(q[1] for q in queries) + 2
    pref = [0]; x = 1
    while len(pref) < need:
        for b in range(x.bit_length()):
            if x >> b & 1:
                pref.append(pref[-1] + b)
        x += 1
    return [pow(2, pref[to + 1] - pref[fr], mod) for fr, to, mod in queries]

for q, exp in examples:
    print('tiền tố:', solve_prefix(q), '| kỳ vọng', exp)

tiền tố: [4] | kỳ vọng [4]
tiền tố: [2, 2] | kỳ vọng [2, 2]


## 8. TRACE từng bước

Trace `E_binary` cho vài `N`, in `x`, `C(x)`, phần dư và kết quả; so với cộng trực tiếp.

In [7]:
arr = [v for x in range(1, 200) for v in powerful(x)]
for N in (0, 1, 3, 4, 6, 13, 50):
    lo, hi = 0, 1
    while C(hi) <= N:
        hi *= 2
    while lo < hi:
        mid = (lo + hi + 1) // 2
        lo, hi = (mid, hi) if C(mid) <= N else (lo, mid - 1)
    x = lo; r = N - C(x)
    direct = sum(v.bit_length() - 1 for v in arr[:N])
    print(f'N = {N:3d}: x = {x:3d}, C(x) = {C(x):3d}, S(x) = {S(x):3d}, dư r = {r} bit thấp của {x + 1} ({bin(x + 1)}) -> E = {E_binary(N)} | trực tiếp {direct}')

N =   0: x =   0, C(x) =   0, S(x) =   0, dư r = 0 bit thấp của 1 (0b1) -> E = 0 | trực tiếp 0
N =   1: x =   1, C(x) =   1, S(x) =   0, dư r = 0 bit thấp của 2 (0b10) -> E = 0 | trực tiếp 0
N =   3: x =   2, C(x) =   2, S(x) =   1, dư r = 1 bit thấp của 3 (0b11) -> E = 1 | trực tiếp 1
N =   4: x =   3, C(x) =   4, S(x) =   2, dư r = 0 bit thấp của 4 (0b100) -> E = 2 | trực tiếp 2
N =   6: x =   4, C(x) =   5, S(x) =   4, dư r = 1 bit thấp của 5 (0b101) -> E = 4 | trực tiếp 4
N =  13: x =   8, C(x) =  13, S(x) =  15, dư r = 0 bit thấp của 9 (0b1001) -> E = 15 | trực tiếp 15
N =  50: x =  22, C(x) =  48, S(x) =  85, dư r = 2 bit thấp của 23 (0b10111) -> E = 86 | trực tiếp 86


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def findProductsOfElements(self, queries: List[List[int]]) -> List[int]:
        def ones(x, b):
            L = x + 1
            return ((L >> (b + 1)) << b) + max(0, (L & ((1 << (b + 1)) - 1)) - (1 << b))

        def count_and_sum(x):
            c = s = 0
            for b in range(x.bit_length() + 1):
                o = ones(x, b)
                c += o
                s += b * o
            return c, s

        def E(N):
            lo, hi = 0, 1
            while count_and_sum(hi)[0] <= N:
                hi *= 2
            while lo < hi:
                mid = (lo + hi + 1) // 2
                if count_and_sum(mid)[0] <= N:
                    lo = mid
                else:
                    hi = mid - 1
            c, e = count_and_sum(lo)
            r = N - c
            y = lo + 1
            b = 0
            while r:
                if y >> b & 1:
                    e += b
                    r -= 1
                b += 1
            return e

        return [pow(2, E(to + 1) - E(fr), mod) for fr, to, mod in queries]

for q, exp in examples:
    got = Solution().findProductsOfElements(q)
    print(got, 'OK' if got == exp else 'SAI')

[4] OK
[2, 2] OK


## 10. Vì sao thuật toán đúng

1. Mọi phần tử là `2^b` ⇒ tích đoạn `= 2^{Σ b}` ⇒ đáp án `= 2^{E(to+1) − E(from)} mod mod`; `pow(2, e, mod)` tính đúng cả khi `mod = 1` (ra 0).
2. **Đếm bit:** công thức chu kỳ đếm đúng số `i ∈ [0, x]` có bit `b` bật (đã kiểm chứng với đếm trực tiếp). Số 0 không có bit nào ⇒ `C`, `S` đúng cho `1..x`.
3. **Định vị:** `C` không giảm; `x` lớn nhất có `C(x) ≤ N` ⇒ các số `1..x` góp trọn, và `N − C(x) < popcount(x + 1)` (nếu không thì `C(x+1) ≤ N`, mâu thuẫn). Mảng mạnh sắp tăng ⇒ các phần tử đầu là bit thấp nhất.

## 11. Phân tích Time / Space Complexity

| Cách | Mỗi truy vấn | Space |
|---|---|---|
| 1. Dựng mảng thật | `O(to)` | `O(to)` |
| 2. Tìm nhị phân + đếm bit | `O(log² X)` | `O(1)` |
| 3. Dựng từng bit | `O(log X)` | `O(1)` |
| 4. Tiền tố (chỉ số nhỏ) | `O(1)` sau `O(maxTo)` tiền xử lý | `O(maxTo)` |

`log X ≈ 50` ⇒ cách 2 khoảng `50 · 50 · 2 = 5.000` bước mỗi `E`, 500 truy vấn ⇒ vài triệu bước.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'binary': solve_binary, 'bitwise': solve_bitwise, 'prefix': solve_prefix,
         'submit': lambda q: Solution().findProductsOfElements(q)}
edge = [([[0, 0, 5]], [1]), ([[0, 0, 1]], [0]), ([[1, 1, 100]], [2]), ([[0, 12, 1000]], [pow(2, 15, 1000)]),
        ([[4, 4, 10], [12, 12, 10]], [4, 8])]
for q, exp in examples + edge:
    assert brute(q) == exp, (q, brute(q))
    for name, f in impls.items():
        assert f(q) == exp, (name, q)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Giải thích: `big_nums[0..12] = [1, 2, 1, 2, 4, 1, 4, 2, 4, 1, 2, 4, 8]` có tổng số mũ `0+1+0+1+2+0+2+1+2+0+1+2+3 = 15` ⇒ `2^15 mod 1000 = 768`. `mod = 1` ⇒ luôn 0. Đoạn một phần tử `[0, 0]` ⇒ `1 mod 5 = 1`.

### 12.2. Đối chiếu ngẫu nhiên với dựng mảng thật

In [10]:
rng = random.Random(3145)
cnt = 0
for _ in range(300):
    qs = []
    for _ in range(rng.randint(1, 5)):
        a = rng.randint(0, 3000); b = rng.randint(a, a + rng.choice([0, 5, 300]))
        qs.append([a, b, rng.randint(1, 10 ** 5)])
    exp = brute(qs)
    for name, f in impls.items():
        assert f(qs) == exp, (name, qs)
    cnt += 1
for N in range(0, 3000):
    assert E_binary(N) == E_bitwise(N)
print('Khớp brute trên', cnt, 'bộ truy vấn; E_binary = E_bitwise với mọi N < 3000.')

Khớp brute trên 300 bộ truy vấn; E_binary = E_bitwise với mọi N < 3000.


### 12.3. Ca lớn: 500 truy vấn, chỉ số tới `10^15`

In [11]:
qs = [[a, a + rng.randint(0, 10 ** 15 - a), rng.randint(1, 10 ** 5)] for a in (rng.randint(0, 10 ** 15) for _ in range(500))]
qs = [[min(a, b), max(a, b) if max(a, b) <= 10 ** 15 else 10 ** 15, m] for a, b, m in qs]
t = time.time(); x = Solution().findProductsOfElements(qs); t1 = time.time() - t
t = time.time(); y = solve_bitwise(qs); t2 = time.time() - t
assert x == y
print(f'500 truy vấn lớn: nộp {t1:.2f}s | dựng từng bit {t2:.2f}s | 5 đáp án đầu {x[:5]}')

500 truy vấn lớn: nộp 0.87s | dựng từng bit 0.02s | 5 đáp án đầu [34596, 47560, 35049, 23986, 31204]


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đánh chỉ số `big_nums` từ 1

In [12]:
def wrong_one_indexed(queries):
    return [pow(2, E_binary(to) - E_binary(fr - 1) if fr >= 1 else E_binary(to), mod) for fr, to, mod in queries]   # SAI

print('sai :', wrong_one_indexed(examples[0][0]), '| đúng:', Solution().findProductsOfElements(examples[0][0]))

sai : [2] | đúng: [4]


**Vì sao sai:** `big_nums[0] = 1` (từ số 1); lệch một chỉ số làm đoạn trượt sang trái.

### Lỗi 2 — Phần dư lấy bit **cao nhất** của `x + 1`

In [13]:
def E_high_bits(N):
    lo, hi = 0, 1
    while C(hi) <= N:
        hi *= 2
    while lo < hi:
        mid = (lo + hi + 1) // 2
        lo, hi = (mid, hi) if C(mid) <= N else (lo, mid - 1)
    r = N - C(lo); e = S(lo); y = lo + 1; b = y.bit_length() - 1
    while r:
        if y >> b & 1:
            e += b; r -= 1                                                # SAI: mảng mạnh sắp TĂNG
        b -= 1
    return e

q = [[0, 5, 1000]]
print('sai :', [pow(2, E_high_bits(6) - E_high_bits(0), 1000)], '| đúng:', Solution().findProductsOfElements(q))

sai : [64] | đúng: [16]


**Vì sao sai:** `big_nums[0..5] = [1, 2, 1, 2, 4, 1]` — `big_nums[5] = 1` là phần tử **đầu** (nhỏ nhất) của mảng mạnh của 5 (`[1, 4]`); lấy bit cao cho `4`.

### Lỗi 3 — Giảm số mũ theo `mod − 1` (Fermat) khi `mod` không nguyên tố

In [14]:
def wrong_fermat(queries):
    return [pow(2, (E_binary(to + 1) - E_binary(fr)) % (mod - 1) if mod > 1 else 0, mod) for fr, to, mod in queries]   # SAI

q = [[0, 20, 8]]
print('sai :', wrong_fermat(q), '| đúng:', Solution().findProductsOfElements(q))

sai : [1] | đúng: [0]


**Vì sao sai:** `2^{e} ≡ 2^{e mod (p−1)}` chỉ đúng khi `p` nguyên tố và không chia hết 2. Với `mod = 8`, `2^e ≡ 0` khi `e ≥ 3`. Python `pow(2, e, mod)` với `e` lớn đã nhanh — không cần giảm số mũ.

### Lỗi 4 — Công thức đếm bit thiếu `max(0, ·)` ở phần dư

In [15]:
def ones_wrong(x, b):
    L = x + 1
    return (L >> (b + 1) << b) + (L & ((1 << (b + 1)) - 1)) - (1 << b)    # SAI: có thể âm

print('ones(4, bit 1) đúng =', ones_bit(4, 1), '| sai =', ones_wrong(4, 1))
print('ones(0, bit 1) đúng =', ones_bit(0, 1), '| sai =', ones_wrong(0, 1))
C_wrong = lambda x: sum(ones_wrong(x, b) for b in range(x.bit_length() + 1)) if x > 0 else 0
print('C(4) đúng =', C(4), '| sai =', C_wrong(4))

ones(4, bit 1) đúng = 2 | sai = 1
ones(0, bit 1) đúng = 0 | sai = -1
C(4) đúng = 5 | sai = 1


**Vì sao sai:** khi phần dư `< 2^b` (chưa tới nửa "bật" của chu kỳ), số bit 1 thêm là 0 chứ không âm. Ví dụ `x = 4`, bit 1: dãy `0..4` có `2, 3` bật bit 1 (đúng 2), nhưng phần dư `5 mod 4 = 1 < 2` làm công thức sai cộng `−1` ⇒ `1`; với `x = 0` thì ra hẳn `−1`. Kéo theo `C(4)` sai.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Tích các lũy thừa của 2 ⇒ cộng số mũ.
- "Phần tử thứ `N` của dãy nối" ⇒ hàm đếm tiền tố + tìm nhị phân.
- Đếm bit 1 của `0..x` theo từng bit bằng chu kỳ `2^{b+1}`.
- `pow(base, exp, mod)` của Python xử lý `exp` khổng lồ.

## 15. Biến thể của bài

1. **LC 3007:** số `x` lớn nhất sao cho tổng "giá" (số bit 1 ở các vị trí chia hết cho `x`) `≤ k` — cùng kỹ thuật đếm bit theo chu kỳ + tìm nhị phân.
2. Tổng (thay vì tích) các phần tử ⇒ cần `Σ 2^b · ones_b` thay vì `Σ b · ones_b`.
3. Mảng mạnh sắp **giảm** ⇒ phần dư lấy bit cao nhất.
4. Lũy thừa của 3 (biểu diễn cơ số 3 với chữ số 0/1/2) ⇒ đếm theo chữ số cơ số 3.

In [16]:
# Biến thể 2: tổng big_nums[from..to] (không mod)
def total_sum(fr, to):
    def T(N):
        lo, hi = 0, 1
        while C(hi) <= N:
            hi *= 2
        while lo < hi:
            mid = (lo + hi + 1) // 2
            lo, hi = (mid, hi) if C(mid) <= N else (lo, mid - 1)
        s = sum(ones_bit(lo, b) << b for b in range(lo.bit_length() + 1)) if lo > 0 else 0
        r = N - C(lo); y = lo + 1; b = 0
        while r:
            if y >> b & 1:
                s += 1 << b; r -= 1
            b += 1
        return s
    return T(to + 1) - T(fr)

arr = [v for x in range(1, 2000) for v in powerful(x)]
for _ in range(300):
    a = rng.randint(0, 5000); b = rng.randint(a, a + 200)
    assert total_sum(a, b) == sum(arr[a:b + 1])
print('Biến thể tổng: khớp cộng trực tiếp.')

Biến thể tổng: khớp cộng trực tiếp.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Mỗi truy vấn | Ghi chú |
|---|---|---|---|
| 1 | Dựng mảng | `O(to)` | Oracle |
| 2 | Tìm nhị phân + đếm bit | `O(log² X)` | **Nộp bài** |
| 3 | Dựng `x` từng bit | `O(log X)` | Nhanh nhất, công thức khối tinh tế |
| 4 | Tiền tố | `O(1)` | Chỉ cho chỉ số nhỏ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[[0, 2, 100]]`.

**Bài 2 (tính tay):** `big_nums[6..9]` và tích mod 10.

**Bài 3 (điền TODO):** hàm `value_at(i)` trả `big_nums[i]` với `i ≤ 10^15`.

**Bài 4 (tự viết lại):** đếm số phần tử bằng đúng `2^t` trong `big_nums[from..to]`.

**Bài 5 (mini challenge):** biến thể 3 — mảng mạnh sắp **giảm dần**; tính tích như đề.

In [17]:
def value_at_todo(i):
    # TODO
    raise NotImplementedError

def count_power_todo(fr, to, t):
    # TODO
    raise NotImplementedError

def descending_version_todo(queries):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `[1, 2, 1]`. (2) Tích `2`. (3) `[2]`.

**Bài 2.** (1) Chỉ số 6..9: `[4, 2, 4, 1]` (từ `5 → [1,4]`, `6 → [2,4]`, `7 → [1,…]`). (2) Tích `32`. (3) `2`.

**Bài 3.** (1) `x` lớn nhất với `C(x) ≤ i`. (2) Phần tử là bit thấp thứ `i − C(x)` (0-index) của `x + 1`. (3) Kiểm bằng mảng thật.

**Bài 4.** (1) Đếm số lần bit `t` bật trong `1..x`: `ones_t(x)`. (2) Tiền tố `P(N)` + phần dư. (3) `P(to+1) − P(from)`.

**Bài 5.** (1) Phần dư lấy bit cao nhất. (2) Phần trọn không đổi. (3) Kiểm bằng mảng thật dựng theo thứ tự giảm.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 3
assert Solution().findProductsOfElements([[0, 2, 100]]) == [2]
assert Solution().findProductsOfElements([[6, 9, 10]]) == [32 % 10]

def locate(i):
    lo, hi = 0, 1
    while C(hi) <= i:
        hi *= 2
    while lo < hi:
        mid = (lo + hi + 1) // 2
        lo, hi = (mid, hi) if C(mid) <= i else (lo, mid - 1)
    return lo, i - C(lo)

def value_at(i):
    x, r = locate(i); y = x + 1; b = 0
    while True:
        if y >> b & 1:
            if r == 0:
                return 1 << b
            r -= 1
        b += 1

arr = [v for x in range(1, 3000) for v in powerful(x)]
assert all(value_at(i) == arr[i] for i in range(0, 15000, 7))
print('Bài 1: [2] | Bài 2: 2 | Bài 3: value_at khớp mảng thật. big_nums[10^15] =', value_at(10 ** 15))

Bài 1: [2] | Bài 2: 2 | Bài 3: value_at khớp mảng thật. big_nums[10^15] = 16384


In [19]:
# Bài 4, 5
def count_power(fr, to, t):
    def P(N):
        x, r = locate(N); c = ones_bit(x, t) if x > 0 else 0; y = x + 1; b = 0
        while r:
            if y >> b & 1:
                c += b == t; r -= 1
            b += 1
        return c
    return P(to + 1) - P(fr)

def descending_version(queries):
    def E(N):
        x, r = locate(N); e = S(x); y = x + 1; b = y.bit_length() - 1
        while r:
            if y >> b & 1:
                e += b; r -= 1
            b -= 1
        return e
    return [pow(2, E(to + 1) - E(fr), mod) for fr, to, mod in queries]

arr_desc = [v for x in range(1, 3000) for v in powerful(x)[::-1]]
for _ in range(300):
    a = rng.randint(0, 8000); b = rng.randint(a, a + 100); t = rng.randint(0, 6); m = rng.randint(1, 1000)
    assert count_power(a, b, t) == arr[a:b + 1].count(1 << t)
    p = 1 % m
    for v in arr_desc[a:b + 1]:
        p = p * v % m
    assert descending_version([[a, b, m]]) == [p]
print('Bài 4 (đếm 2^t) và Bài 5 (mảng mạnh giảm dần) khớp mảng thật.')

Bài 4 (đếm 2^t) và Bài 5 (mảng mạnh giảm dần) khớp mảng thật.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
ones(x, b) = ((x+1) >> (b+1) << b) + max(0, ((x+1) & (2^(b+1)-1)) - 2^b)
C(x) = Σ ones(x, b);  S(x) = Σ b·ones(x, b)
E(N): x = max{C(x) <= N}; e = S(x) + vị trí r = N − C(x) bit thấp nhất của x+1
ans = pow(2, E(to+1) − E(from), mod)
```

**Checklist:**

- [ ] Chỉ số `big_nums` từ 0 (`E(to+1) − E(from)`)?
- [ ] Phần dư lấy bit **thấp** nhất?
- [ ] `max(0, ·)` trong công thức đếm bit?
- [ ] Không giảm số mũ theo Fermat; `mod = 1` cho 0?